In [ ]:
import os
if not os.path.exists('/usr/bin/unrar'):
  !apt-get update
  !apt-get install -y unrar-free

In [ ]:
!unrar x /content/train_parquet.rar


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/train_parquet.rar

Creating    limpieza_minima                                           OK
Creating    limpieza_minima/train_parquet                             OK
Extracting  limpieza_minima/train_parquet/part_0.parquet                   3%  6%  7%  OK 
Extracting  limpieza_minima/train_parquet/part_1.parquet                  10% 14%  OK 
Extracting  limpieza_minima/train_parquet/part_10.parquet                 18% 21% 22%  OK 
Extracting  limpieza_minima/train_parquet/part_11.parquet                 25% 29%  OK 
Extracting  limpieza_minima/train_parquet/part_12.parquet                 32% 36%  OK 
Extracting  limpieza_minima/train_parquet/part_13.parquet                 40% 41%  OK 
Extracting  limpieza_minima/train_parquet/part_2.parquet                  44% 48%  OK 
Extracting  limpie

In [ ]:
!pip install spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 90.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


In [ ]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 49.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import os
import spacy
import re
import time

In [ ]:
# Nombre de la subcarpeta para este experimento
EXPERIMENT_FOLDER = "Exp03_Solo_Puntuacion"

# Interruptores de Ablación (Solo Lematización activada)
USE_LEMMATIZATION = False
DROP_STOPWORDS = False
DROP_PUNCTUATION = True
NORMALIZE_ELONGATION = False

# Rutas
INPUT_TRAIN_PREFIX = "/content/limpieza_minima/train_parquet"
OUTPUT_TRAIN_PREFIX = f"Spacy/{EXPERIMENT_FOLDER}/train_parquet"

TEXT_COLUMN = "clean_text"

In [ ]:
print("Cargando modelo de spaCy (Optimizado)...")

nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])

Cargando modelo de spaCy (Optimizado)...


In [ ]:
def ablation_clean(text: str) -> str:
    if pd.isna(text):
        return ""

    text = str(text).lower()

    if NORMALIZE_ELONGATION:
        text = re.sub(r'(.)\1{2,}', r'\1', text)

    doc = nlp(text)
    tokens = []

    for token in doc:
        if DROP_STOPWORDS and token.is_stop: continue
        if DROP_PUNCTUATION and token.is_punct: continue

        word = token.lemma_ if USE_LEMMATIZATION else token.text
        tokens.append(word)

    return " ".join(tokens)

In [ ]:
def run_etl():
    # Crear carpeta de salida si no existe
    os.makedirs(OUTPUT_TRAIN_PREFIX, exist_ok=True)

    files = [
        os.path.join(INPUT_TRAIN_PREFIX, f)
        for f in os.listdir(INPUT_TRAIN_PREFIX)
        if f.endswith(".parquet")
    ]

    if not files:
        print(f"No se encontraron archivos en {INPUT_TRAIN_PREFIX}")
        return

    print(f"Iniciando procesamiento para: {EXPERIMENT_FOLDER}")
    start_time = time.time()

    for file in files:
        filename = os.path.basename(file)
        output_path = os.path.join(OUTPUT_TRAIN_PREFIX, filename)

        # Validar si ya se procesó
        if os.path.exists(output_path):
            print(f"  ⏭️ Partición {filename} ya existe. Saltando...")
            continue

        print(f"Procesando: {filename}...")
        df = pd.read_parquet(file)

        # Aplicamos la limpieza
        df["ablation_text"] = df[TEXT_COLUMN].apply(ablation_clean)

        # Guardamos en carpeta local
        df.to_parquet(
            output_path,
            engine="pyarrow",
            index=False,
            compression="snappy"
        )

        print(f" Guardado en {output_path}")

    elapsed = time.time() - start_time
    print(f"Experimento '{EXPERIMENT_FOLDER}' completado en {elapsed/60:.2f} minutos.")


# ¡Ejecutar!
run_etl()

Iniciando procesamiento para: Exp03_Solo_Puntuacion
Procesando: part_10.parquet...
 Guardado en Spacy/Exp03_Solo_Puntuacion/train_parquet/part_10.parquet
Procesando: part_8.parquet...
 Guardado en Spacy/Exp03_Solo_Puntuacion/train_parquet/part_8.parquet
Procesando: part_12.parquet...
 Guardado en Spacy/Exp03_Solo_Puntuacion/train_parquet/part_12.parquet
Procesando: part_11.parquet...
 Guardado en Spacy/Exp03_Solo_Puntuacion/train_parquet/part_11.parquet
Procesando: part_13.parquet...
 Guardado en Spacy/Exp03_Solo_Puntuacion/train_parquet/part_13.parquet
Procesando: part_4.parquet...
 Guardado en Spacy/Exp03_Solo_Puntuacion/train_parquet/part_4.parquet
Procesando: part_5.parquet...
 Guardado en Spacy/Exp03_Solo_Puntuacion/train_parquet/part_5.parquet
Procesando: part_1.parquet...
 Guardado en Spacy/Exp03_Solo_Puntuacion/train_parquet/part_1.parquet
Procesando: part_9.parquet...
 Guardado en Spacy/Exp03_Solo_Puntuacion/train_parquet/part_9.parquet
Procesando: part_2.parquet...
 Guardado 

In [ ]:
!zip -r Exp03_Solo_Puntuacion.zip /content/Exp03_Solo_Puntuacion

  adding: content/Exp03_Solo_Puntuacion/ (stored 0%)
  adding: content/Exp03_Solo_Puntuacion/train_parquet/ (stored 0%)
  adding: content/Exp03_Solo_Puntuacion/train_parquet/part_10.parquet (deflated 14%)
  adding: content/Exp03_Solo_Puntuacion/train_parquet/part_8.parquet (deflated 14%)
  adding: content/Exp03_Solo_Puntuacion/train_parquet/part_12.parquet (deflated 14%)
  adding: content/Exp03_Solo_Puntuacion/train_parquet/part_11.parquet (deflated 14%)
  adding: content/Exp03_Solo_Puntuacion/train_parquet/part_13.parquet (deflated 14%)
  adding: content/Exp03_Solo_Puntuacion/train_parquet/part_4.parquet (deflated 14%)
  adding: content/Exp03_Solo_Puntuacion/train_parquet/part_5.parquet (deflated 14%)
  adding: content/Exp03_Solo_Puntuacion/train_parquet/part_1.parquet (deflated 14%)
  adding: content/Exp03_Solo_Puntuacion/train_parquet/part_9.parquet (deflated 14%)
  adding: content/Exp03_Solo_Puntuacion/train_parquet/part_2.parquet (deflated 14%)
  adding: content/Exp03_Solo_Puntuac